# 🔬 SparseGPT + INT8 Quantization + CodeCarbon
### TinyLlama-1.1B on Google Colab

This notebook implements:
- **SparseGPT** (Frantar & Alistarh, 2023) — Hessian-guided one-shot pruning
- **INT8 Quantization** — post-pruning absmax symmetric quantization
- **CodeCarbon** — CO₂ emission tracking for each pipeline phase

**Pipeline:** Baseline → SparseGPT 50% sparse → INT8 quantize → Evaluate all

---
> ⚡ **Before running:** `Runtime → Change runtime type → T4 GPU`

In [1]:
# @title Cell 1 — Install Dependencies
!pip install -q codecarbon
# ============================================================
# SparseGPT + INT8 Quantization + CodeCarbon Emissions Tracker
# TinyLlama-1.1B on Google Colab
# ============================================================

In [2]:
# @title Cell 2 — Imports & GPU Check

# @title Install Dependencies
"""
Run this cell first. Restart runtime after installation.
"""
# !pip install transformers datasets accelerate torch bitsandbytes \
#              codecarbon matplotlib seaborn tabulate scipy -q

'\nRun this cell first. Restart runtime after installation.\n'

In [3]:
# @title Cell 3 — SparseGPT Implementation

# @title Imports & Setup
import copy, math, time, os, json
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from codecarbon import EmissionsTracker

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {device}")
if device == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found — go to Runtime > Change runtime type > T4 GPU")

Device  : cuda
GPU     : Tesla T4
VRAM    : 15.6 GB


In [4]:
# @title Cell 4 — INT8 Quantization Helpers

# @title SparseGPT Implementation

"""
SparseGPT (Frantar & Alistarh, 2023) — key insight:
  Magnitude pruning ignores the Hessian (curvature of the loss).
  SparseGPT uses the *inverse Hessian* of each layer to compensate
  for the damage caused by zeroing a weight — updating remaining
  weights to minimise reconstruction error.

  H = 2 * X @ X.T          (layer input covariance)
  H_inv = inverse(H + λI)   (regularised inverse Hessian)

  For each weight w_ij being zeroed:
      error    = w_ij² / H_inv[j,j]
      update   = w_ij / H_inv[j,j] * H_inv[:, j]  (propagate error)

This is a *one-shot* method: no gradient updates, no fine-tuning.
"""

PRUNABLE_LAYER_NAMES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ── Hessian accumulator ──────────────────────────────────────────────

class HessianHook:
    def __init__(self, layer: nn.Linear):
        self.layer = layer
        n = layer.weight.shape[1]
        self.device = layer.weight.device          # ← capture layer's device
        self.H = torch.zeros(n, n, dtype=torch.float32, device=self.device)  # ← put H on same device
        self.n_samples = 0
        self._handle = layer.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        x = inp[0].detach().float()                # already on correct device
        if x.dim() == 3:
            x = x.reshape(-1, x.shape[-1])
        self.H += 2.0 * x.T @ x
        self.n_samples += x.shape[0]

    def remove(self):
        self._handle.remove()

    def get_hessian(self, damping: float = 0.01) -> torch.Tensor:
        H = self.H.clone()
        if self.n_samples > 0:
            H /= self.n_samples
        diag_mean = H.diagonal().mean()
        H += damping * diag_mean * torch.eye(H.shape[0], device=self.device)  # ← eye on same device
        return H


# ── Per-layer SparseGPT solver ───────────────────────────────────────

def sparsegpt_layer(
    weight: torch.Tensor,
    H_inv: torch.Tensor,
    sparsity: float,
    block_size: int = 128,
) -> Tuple[torch.Tensor, torch.Tensor]:
    dev = weight.device                            # ← capture device
    W = weight.clone().float().to(dev)             # ← ensure W is on dev
    H_inv = H_inv.float().to(dev)                  # ← move H_inv to same device
    out_dim, in_dim = W.shape
    mask = torch.ones_like(W, device=dev)          # ← mask on same device

    for block_start in range(0, in_dim, block_size):
        block_end = min(block_start + block_size, in_dim)
        W_block = W[:, block_start:block_end].clone()
        H_inv_block = H_inv[block_start:block_end, block_start:block_end]

        for col in range(block_end - block_start):
            global_col = block_start + col
            h_diag = H_inv[global_col, global_col].clamp(min=1e-8)
            scores = (W[:, global_col] ** 2) / h_diag
            n_prune = max(0, round(sparsity * out_dim) - int(
                (mask[:, block_start:global_col] == 0).any(dim=1).sum().item()))
            if n_prune <= 0:
                continue
            topk = torch.topk(scores, n_prune, largest=False).indices
            mask[topk, global_col] = 0.0
            err = W[:, global_col] / h_diag
            err[topk] = 0.0
            if col + 1 < block_end - block_start:
                W_block[:, col+1:] -= torch.outer(
                    err, H_inv[global_col, block_start+col+1:block_end]
                )

        W[:, block_start:block_end] = W_block

    W.mul_(mask)
    return W, mask

# Emission tracker helper

def make_tracker(project_name: str) -> EmissionsTracker:
    """Create an EmissionsTracker, ensuring the output dir exists first."""
    os.makedirs("./emissions", exist_ok=True)
    return EmissionsTracker(
        project_name=project_name,
        output_dir="./emissions",
        log_level="error",
        save_to_file=True,
    )

# ── Full model SparseGPT ─────────────────────────────────────────────

@dataclass
class SparseGPTStats:
    sparsity_target: float
    layer_stats: Dict[str, dict] = field(default_factory=dict)

    @property
    def actual_sparsity(self):
        total = sum(s["total"] for s in self.layer_stats.values())
        zeros = sum(s["zeros"] for s in self.layer_stats.values())
        return zeros / total if total > 0 else 0.0

    def summary(self):
        total = sum(s["total"] for s in self.layer_stats.values())
        zeros = sum(s["zeros"] for s in self.layer_stats.values())
        print(f"\n{'='*55}")
        print(f"  SparseGPT — {self.sparsity_target*100:.0f}% Target Sparsity")
        print(f"{'='*55}")
        print(f"  Actual sparsity : {self.actual_sparsity*100:.2f}%")
        print(f"  Zeroed weights  : {zeros:,} / {total:,}")
        print(f"  Layers pruned   : {len(self.layer_stats)}")
        print(f"{'='*55}")


class SparseGPTPruner:
    """
    SparseGPT pruning for TinyLlama.

    Phase 1 — Calibration: run `n_samples` forward passes to accumulate
              per-layer input covariance (Hessian).
    Phase 2 — Pruning: for each layer, invert the Hessian and apply
              the SparseGPT solver.

    Args:
        model       : The TinyLlama model
        tokenizer   : Matching tokenizer
        sparsity    : Fraction of weights to zero (e.g. 0.5)
        n_samples   : Calibration samples (more = better Hessian estimate)
        seq_len     : Token sequence length for calibration
        damping     : Hessian regularisation (prevents near-singular inversion)
    """

    def __init__(
        self,
        model: nn.Module,
        tokenizer,
        sparsity: float,
        n_samples: int = 128,
        seq_len: int = 512,
        damping: float = 0.01,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.sparsity = sparsity
        self.n_samples = n_samples
        self.seq_len = seq_len
        self.damping = damping
        self._hooks: Dict[str, HessianHook] = {}
        self._masks: Dict[str, torch.Tensor] = {}

    def _get_calibration_data(self) -> List[torch.Tensor]:
        """Sample random sequences from WikiText-2 for calibration."""
        dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        text = "\n\n".join(dataset["text"])
        enc = self.tokenizer(text, return_tensors="pt").input_ids[0]

        samples = []
        for _ in range(self.n_samples):
            start = torch.randint(0, len(enc) - self.seq_len, (1,)).item()
            samples.append(enc[start : start + self.seq_len].unsqueeze(0))
        return samples

    def _attach_hooks(self):
        """Attach Hessian accumulation hooks to all prunable layers."""
        for name, module in self.model.named_modules():
            if isinstance(module, nn.Linear) and any(name.endswith(ln) for ln in PRUNABLE_LAYER_NAMES):
                self._hooks[name] = HessianHook(module)

    def _remove_hooks(self):
        for hook in self._hooks.values():
            hook.remove()
        self._hooks.clear()

    def _calibrate(self, samples: List[torch.Tensor]):
        """Run calibration forward passes (no gradients)."""
        self.model.eval()
        dev = next(self.model.parameters()).device
        with torch.no_grad():
            for i, ids in enumerate(samples):
                self.model(ids.to(dev))
                if (i + 1) % 32 == 0:
                    print(f"    Calibration: {i+1}/{len(samples)} samples")

    def prune(self) -> SparseGPTStats:
        """
        Full SparseGPT pipeline:
          1. Attach hooks
          2. Calibration forward passes
          3. For each layer: invert Hessian → SparseGPT solve → update weights
          4. Remove hooks
        """
        print("  [1/3] Collecting calibration data...")
        samples = self._get_calibration_data()

        print("  [2/3] Calibration forward passes...")
        self._attach_hooks()
        self._calibrate(samples)
        self._remove_hooks()

        print("  [3/3] Applying SparseGPT layer by layer...")
        stats = SparseGPTStats(sparsity_target=self.sparsity)

        for name, hook in list(self._hooks.items()):
            pass  # hooks already removed, kept for reference

        # Re-iterate to apply pruning in order
        for name, module in self.model.named_modules():
            if not (isinstance(module, nn.Linear) and
                    any(name.endswith(ln) for ln in PRUNABLE_LAYER_NAMES)):
                continue

            hook = self._hooks.get(name)
            if hook is None:
                # Hooks removed; rebuild Hessian from scratch isn't feasible here
                # In practice you'd store H before removing hooks
                continue

        # ── Proper approach: collect H while hooks are live ──────────
        # Re-run with persistent storage
        hessians: Dict[str, torch.Tensor] = {}
        self._attach_hooks()
        self._calibrate(samples)
        for name, hook in self._hooks.items():
            hessians[name] = hook.get_hessian(self.damping).cpu()
        self._remove_hooks()

        # Now prune using stored Hessians
        n_layers = len(hessians)
        for i, (name, module) in enumerate(
            (n, m) for n, m in self.model.named_modules()
            if n in hessians
        ):
            dev = module.weight.device                     # ← get the layer's device
            H = hessians[name].to(dev).to(torch.float32)  # ← move H to GPU

            try:
                H_inv = torch.linalg.inv(H)
            except torch.linalg.LinAlgError:
                H_inv = torch.linalg.pinv(H)
            # H_inv is now on `dev` (cuda:0), matching the weight

            W_orig = module.weight.data.to(dev)            # ← keep weight on GPU
            W_pruned, mask = sparsegpt_layer(W_orig, H_inv, self.sparsity)

            module.weight.data = W_pruned.to(module.weight.dtype)
            self._masks[name] = mask.cpu()                 # ← store mask on CPU to save VRAM

            zeros = int((W_pruned == 0).sum().item())
            stats.layer_stats[name] = {
                "total": W_pruned.numel(),
                "zeros": zeros,
                "threshold": "hessian-guided",
            }
            if (i + 1) % 10 == 0:
                print(f"    Pruned {i+1}/{n_layers} layers...")

            # Free GPU memory as we go
            del H, H_inv, W_orig, W_pruned, mask
            torch.cuda.empty_cache()

        return stats

    def get_masks(self):
        return self._masks


print("✅ SparseGPT classes defined")

✅ SparseGPT classes defined


In [5]:
# @title Cell 5 — Load TinyLlama

# @title INT8 Quantization (post-pruning)

"""
INT8 Quantization via bitsandbytes (absmax symmetric quantization).

  scale = max(|W|) / 127
  W_int8 = round(W / scale).clamp(-128, 127)
  W_dequant = W_int8 * scale

Applied AFTER pruning so the zero pattern is preserved.
The effective compression: float16 weights → int8 = 2x memory reduction.
Combined with 50% sparsity: ~4x total reduction vs dense float16.
"""

def quantize_layer_int8(
    module: nn.Linear,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Symmetric per-tensor absmax INT8 quantization of a linear layer weight.

    Returns:
        w_int8      : quantized int8 weight (stored as int8)
        scale       : float32 scale factor
        w_dequant   : dequantized float weight (for quality check)
    """
    W = module.weight.data.float()
    scale = W.abs().max() / 127.0
    scale = scale.clamp(min=1e-8)

    W_int8 = W.div(scale).round().clamp(-128, 127).to(torch.int8)
    W_dequant = W_int8.float() * scale

    return W_int8, scale, W_dequant


def apply_int8_quantization(model: nn.Module) -> Dict[str, dict]:
    quant_info = {}
    for name, module in model.named_modules():
        if not (isinstance(module, nn.Linear) and
                any(name.endswith(ln) for ln in PRUNABLE_LAYER_NAMES)):
            continue

        dev = module.weight.device                 # ← capture device
        W_original = module.weight.data.float()    # stays on dev
        scale = W_original.abs().max() / 127.0
        scale = scale.clamp(min=1e-8)

        W_int8 = W_original.div(scale).round().clamp(-128, 127).to(torch.int8)
        W_dequant = W_int8.float().to(dev) * scale # ← dequant stays on dev

        module.weight.data = W_dequant.to(module.weight.dtype)

        quant_error = (W_original - W_dequant).abs().mean().item()
        quant_info[name] = {
            "scale": scale.item(),
            "quant_error_mae": quant_error,
        }
    return quant_info


def model_memory_mb(model: nn.Module, assume_int8: bool = False) -> float:
    """Estimate model memory in MB."""
    total_bytes = 0
    for p in model.parameters():
        bytes_per_elem = 1 if assume_int8 else p.element_size()
        total_bytes += p.numel() * bytes_per_elem
    return total_bytes / (1024 ** 2)


print("✅ INT8 quantization helpers defined")

✅ INT8 quantization helpers defined


In [6]:
# @title Cell 6 — Perplexity Helper

# @title Load TinyLlama

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading {MODEL_ID}...")
print("(~2.2 GB download on first run)\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Load in float16 on GPU, float32 on CPU
dtype = torch.float16 if device == "cuda" else torch.float32

baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map=device,
)
baseline_model.eval()

n_params = sum(p.numel() for p in baseline_model.parameters())
print(f"✅ Loaded: {n_params/1e6:.1f}M parameters")
print(f"   Memory footprint: {model_memory_mb(baseline_model):.0f} MB")

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...
(~2.2 GB download on first run)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Loaded: 1100.0M parameters
   Memory footprint: 2098 MB


In [7]:
# @title Cell 7 — Run Full Pipeline (with Emissions Tracking)

# @title Perplexity Helper

def compute_perplexity(m, tok, label="", max_tokens=2048, verbose=True):
    """Sliding-window perplexity on WikiText-2 test set."""
    m.eval()
    dev = next(m.parameters()).device
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(dataset["text"])
    ids = tok(text, return_tensors="pt").input_ids[0][:max_tokens].to(dev)

    stride, window, nlls = 512, 1024, []
    prev_end = 0
    with torch.no_grad():
        for begin in range(0, len(ids), stride):
            end = min(begin + window, len(ids))
            target_len = end - prev_end
            chunk = ids[begin:end].unsqueeze(0)
            labels = chunk.clone()
            labels[:, :-target_len] = -100
            loss = m(chunk, labels=labels).loss
            nlls.append(loss.float() * target_len)
            prev_end = end
            if end == len(ids):
                break

    ppl = math.exp(torch.stack(nlls).sum().item() / len(ids))
    if verbose:
        print(f"  {label:>30}  →  PPL = {ppl:.2f}")
    return ppl

In [8]:
# @title Cell 8 — Results Summary Table

# @title Run Full Pipeline with Emissions Tracking

"""
Pipeline:
  Baseline  →  SparseGPT 50%  →  INT8 quantization  →  Evaluate all

CodeCarbon tracks CO₂ emissions for each phase.
All results are collected into `pipeline_results` dict.
"""

SPARSITY = 0.50          # Change to 0.25 or 0.75 as needed
N_CALIB_SAMPLES = 64     # Increase to 128 for better Hessian estimate
MAX_EVAL_TOKENS = 1024   # Increase to 2048 for more accurate perplexity

pipeline_results = {}

# ── Phase 0: Baseline perplexity ─────────────────────────────────────
print("\n" + "="*60)
print("  PHASE 0: Baseline Evaluation")
print("="*60)


tracker  = make_tracker("tinyllama_baseline")
tracker.start()
t0 = time.time()

baseline_ppl = compute_perplexity(
    baseline_model, tokenizer, "Baseline (dense float16)",
    max_tokens=MAX_EVAL_TOKENS
)

tracker  = make_tracker("tinyllama_baseline")
baseline_time = time.time() - t0
baseline_emissions = tracker.stop()


pipeline_results["baseline"] = {
    "perplexity": baseline_ppl,
    "memory_mb": model_memory_mb(baseline_model),
    "eval_time_s": baseline_time,
    "co2_kg": baseline_emissions or 0.0,
}
print(f"  ⏱  Eval time : {baseline_time:.1f}s")
print(f"  🌱 CO₂ emitted: {(baseline_emissions or 0)*1000:.4f} g")


# ── Phase 1: SparseGPT pruning ────────────────────────────────────────
print("\n" + "="*60)
print(f"  PHASE 1: SparseGPT Pruning at {int(SPARSITY*100)}% Sparsity")
print("="*60)

tracker2 = make_tracker("tinyllama_sparsegpt")

tracker2.start()
t1 = time.time()

# Work on a deep copy so baseline is preserved
sparse_model = copy.deepcopy(baseline_model)

pruner = SparseGPTPruner(
    model=sparse_model,
    tokenizer=tokenizer,
    sparsity=SPARSITY,
    n_samples=N_CALIB_SAMPLES,
    seq_len=512,
    damping=0.01,
)
sparse_stats = pruner.prune()
sparse_stats.summary()

# Evaluate pruned model
sparse_ppl = compute_perplexity(
    sparse_model, tokenizer,
    f"SparseGPT {int(SPARSITY*100)}% sparse",
    max_tokens=MAX_EVAL_TOKENS,
)

prune_time = time.time() - t1
prune_emissions = tracker2.stop()

pipeline_results["sparsegpt"] = {
    "perplexity": sparse_ppl,
    "actual_sparsity": sparse_stats.actual_sparsity,
    "memory_mb": model_memory_mb(sparse_model),
    "prune_time_s": prune_time,
    "co2_kg": prune_emissions or 0.0,
}
print(f"  ⏱  Total time : {prune_time:.1f}s")
print(f"  🌱 CO₂ emitted: {(prune_emissions or 0)*1000:.4f} g")


# ── Phase 2: INT8 Quantization ────────────────────────────────────────
print("\n" + "="*60)
print("  PHASE 2: INT8 Quantization (post-pruning)")
print("="*60)

tracker3 = make_tracker("tinyllama_int8")

tracker3.start()
t2 = time.time()

# Quantize the already-pruned model
quant_info = apply_int8_quantization(sparse_model)
avg_quant_error = np.mean([v["quant_error_mae"] for v in quant_info.values()])
print(f"  Avg quantization MAE: {avg_quant_error:.6f}")
print(f"  Layers quantized    : {len(quant_info)}")

# Evaluate pruned + quantized model
quantized_ppl = compute_perplexity(
    sparse_model, tokenizer,
    f"SparseGPT {int(SPARSITY*100)}% + INT8",
    max_tokens=MAX_EVAL_TOKENS,
)

quant_time = time.time() - t2
quant_emissions = tracker3.stop()

pipeline_results["sparsegpt_int8"] = {
    "perplexity": quantized_ppl,
    "actual_sparsity": sparse_stats.actual_sparsity,
    "memory_mb_simulated": model_memory_mb(sparse_model, assume_int8=True),
    "avg_quant_error": avg_quant_error,
    "quant_time_s": quant_time,
    "co2_kg": quant_emissions or 0.0,
}
print(f"  ⏱  Quant time : {quant_time:.1f}s")
print(f"  🌱 CO₂ emitted: {(quant_emissions or 0)*1000:.4f} g")

print("\n✅ Pipeline complete!")

[codecarbon WARNING @ 02:43:26] Multiple instances of codecarbon are allowed to run at the same time.



  PHASE 0: Baseline Evaluation


Token indices sequence length is longer than the specified maximum sequence length for this model (341469 > 2048). Running this sequence through the model will result in indexing errors
[codecarbon ERROR @ 02:43:33] You first need to start the tracker.


        Baseline (dense float16)  →  PPL = 5.03
  ⏱  Eval time : 5.9s
  🌱 CO₂ emitted: 0.0000 g

  PHASE 1: SparseGPT Pruning at 50% Sparsity
  [1/3] Collecting calibration data...
  [2/3] Calibration forward passes...
    Calibration: 32/64 samples
    Calibration: 64/64 samples
  [3/3] Applying SparseGPT layer by layer...
    Calibration: 32/64 samples
    Calibration: 64/64 samples
    Pruned 10/154 layers...
    Pruned 20/154 layers...
    Pruned 30/154 layers...
    Pruned 40/154 layers...
    Pruned 50/154 layers...
    Pruned 60/154 layers...
    Pruned 70/154 layers...
    Pruned 80/154 layers...
    Pruned 90/154 layers...
    Pruned 100/154 layers...
    Pruned 110/154 layers...
    Pruned 120/154 layers...
    Pruned 130/154 layers...
    Pruned 140/154 layers...
    Pruned 150/154 layers...

  SparseGPT — 50% Target Sparsity
  Actual sparsity : 0.39%
  Zeroed weights  : 3,785,345 / 968,884,224
  Layers pruned   : 154
            SparseGPT 50% sparse  →  PPL = 5.07
  ⏱  Tota

In [9]:
# @title Cell 9 — Visualizations

# @title Print Results Summary

print("\n" + "="*70)
print(f"  {'Stage':<30} {'PPL':>8} {'Mem (MB)':>10} {'CO₂ (mg)':>12}")
print("="*70)

rows = [
    ("Baseline (dense fp16)",
     pipeline_results["baseline"]["perplexity"],
     pipeline_results["baseline"]["memory_mb"],
     pipeline_results["baseline"]["co2_kg"] * 1e6),

    (f"SparseGPT {int(SPARSITY*100)}% sparse",
     pipeline_results["sparsegpt"]["perplexity"],
     pipeline_results["sparsegpt"]["memory_mb"],
     pipeline_results["sparsegpt"]["co2_kg"] * 1e6),

    (f"SparseGPT {int(SPARSITY*100)}% + INT8",
     pipeline_results["sparsegpt_int8"]["perplexity"],
     pipeline_results["sparsegpt_int8"]["memory_mb_simulated"],
     pipeline_results["sparsegpt_int8"]["co2_kg"] * 1e6),
]

for name, ppl, mem, co2 in rows:
    print(f"  {name:<30} {ppl:>8.2f} {mem:>10.0f} {co2:>12.4f}")

print("="*70)

base_ppl = rows[0][1]
for name, ppl, mem, co2 in rows[1:]:
    delta = ((ppl - base_ppl) / base_ppl) * 100
    print(f"  {name:<30}  PPL Δ: {delta:+.1f}%  Mem: {mem/rows[0][2]*100:.0f}% of baseline")


  Stage                               PPL   Mem (MB)     CO₂ (mg)
  Baseline (dense fp16)              5.03       2098       0.0000
  SparseGPT 50% sparse               5.07       2098     687.6209
  SparseGPT 50% + INT8               5.19       1049      11.9056
  SparseGPT 50% sparse            PPL Δ: +0.9%  Mem: 100% of baseline
  SparseGPT 50% + INT8            PPL Δ: +3.1%  Mem: 50% of baseline


In [1]:
# @title Cell 10 — Emissions Deep Dive

# @title Plots: Perplexity, Memory, Emissions, Weight Distribution

COLORS = {
    "Baseline": "#5B9BD5",
    f"SparseGPT {int(SPARSITY*100)}%": "#70AD47",
    f"Sparse+INT8": "#ED7D31",
}

fig = plt.figure(figsize=(16, 12))
fig.suptitle("TinyLlama: SparseGPT + INT8 Quantization Analysis", fontsize=15, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

labels = [r[0] for r in rows]
short_labels = ["Baseline", f"Sparse\n{int(SPARSITY*100)}%", "Sparse\n+INT8"]
bar_colors = ["#5B9BD5", "#70AD47", "#ED7D31"]

# Plot 1: Perplexity
ax1 = fig.add_subplot(gs[0, 0])
ppls = [r[1] for r in rows]
bars = ax1.bar(short_labels, ppls, color=bar_colors, width=0.5, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, ppls):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f"{val:.1f}", ha="center", fontsize=10, fontweight="bold")
ax1.set_title("Perplexity ↓", fontweight="bold")
ax1.set_ylabel("Perplexity (lower = better)")
ax1.set_ylim(0, max(ppls) * 1.2)
ax1.spines[["top", "right"]].set_visible(False)

# Plot 2: Memory
ax2 = fig.add_subplot(gs[0, 1])
mems = [r[2] for r in rows]
bars = ax2.bar(short_labels, mems, color=bar_colors, width=0.5, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, mems):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f"{val:.0f}", ha="center", fontsize=10, fontweight="bold")
ax2.set_title("Memory Footprint ↓", fontweight="bold")
ax2.set_ylabel("Estimated size (MB)")
ax2.set_ylim(0, max(mems) * 1.25)
ax2.spines[["top", "right"]].set_visible(False)

# Plot 3: CO2 Emissions (milligrams)
ax3 = fig.add_subplot(gs[0, 2])
co2s = [r[3] for r in rows]
bars = ax3.bar(short_labels, co2s, color=bar_colors, width=0.5, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, co2s):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(co2s)*0.02,
             f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")
ax3.set_title("CO₂ Emissions per eval ↓", fontweight="bold")
ax3.set_ylabel("CO₂ (milligrams)")
ax3.spines[["top", "right"]].set_visible(False)

# Plot 4: Weight histogram comparison (baseline vs sparse)
ax4 = fig.add_subplot(gs[1, :2])
for model_pair, color, label in [
    (baseline_model, "#5B9BD5", "Baseline"),
    (sparse_model, "#ED7D31", f"SparseGPT {int(SPARSITY*100)}% + INT8"),
]:
    weights = []
    for name, module in model_pair.named_modules():
        if isinstance(module, nn.Linear) and any(name.endswith(ln) for ln in PRUNABLE_LAYER_NAMES):
            weights.append(module.weight.data.float().cpu().numpy().flatten())
    w = np.concatenate(weights)
    ax4.hist(w, bins=400, range=(-0.25, 0.25), density=True,
             alpha=0.55, color=color, label=f"{label} (zeros: {(w==0).mean()*100:.1f}%)")

ax4.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax4.set_title("Weight Distribution: Baseline vs Sparse+INT8", fontweight="bold")
ax4.set_xlabel("Weight value")
ax4.set_ylabel("Density")
ax4.legend()
ax4.spines[["top", "right"]].set_visible(False)

# Plot 5: Quantization error per layer (sample of layers)
ax5 = fig.add_subplot(gs[1, 2])
layer_short = [k.split(".")[-2] + "." + k.split(".")[-1] for k in list(quant_info.keys())[:15]]
errors = [quant_info[k]["quant_error_mae"] for k in list(quant_info.keys())[:15]]
ax5.barh(layer_short, errors, color="#70AD47", alpha=0.85)
ax5.set_title("INT8 Quant Error per Layer\n(MAE, first 15)", fontweight="bold")
ax5.set_xlabel("Mean Absolute Error")
ax5.spines[["top", "right"]].set_visible(False)

plt.savefig("sparsegpt_int8_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved as sparsegpt_int8_results.png")

NameError: name 'SPARSITY' is not defined

In [ ]:
# @title Cell 11 — Text Generation Comparison

# @title Emissions Analysis

print("\n🌱 Emissions Summary")
print("="*55)
total_co2_mg = sum(r[3] for r in rows)
print(f"  Total CO₂ this session : {total_co2_mg:.4f} mg")
print(f"  Equivalent to driving  : {total_co2_mg/1e6 / 0.192 * 1000:.4f} meters in avg car")
print(f"  (1g CO₂ ≈ 5.2m driven at ~192g CO₂/km)")
print()

# Load codecarbon CSV if available
emissions_dir = "./emissions"
csv_files = [f for f in os.listdir(emissions_dir) if f.endswith(".csv")] if os.path.exists(emissions_dir) else []
if csv_files:
    import csv
    print("  Raw CodeCarbon data:")
    for fname in sorted(csv_files)[-3:]:
        with open(os.path.join(emissions_dir, fname)) as f:
            rows_csv = list(csv.DictReader(f))
            if rows_csv:
                last = rows_csv[-1]
                print(f"  [{last.get('project_name','?')}]")
                print(f"    duration   : {float(last.get('duration',0)):.1f}s")
                print(f"    emissions  : {float(last.get('emissions',0))*1e6:.4f} mg CO₂")
                print(f"    energy_CPU : {float(last.get('cpu_energy',0))*1000:.4f} Wh")
                print(f"    energy_GPU : {float(last.get('gpu_energy',0))*1000:.4f} Wh")
                print()

In [ ]:
# @title Cell 12 — Save Results to JSON

# @title Compare Generated Text

def generate(m, tok, prompt, max_new_tokens=120):
    fmt = f"<|system|>\nYou are a helpful assistant.</s>\n<|user|>\n{prompt}</s>\n<|assistant|>\n"
    inp = tok(fmt, return_tensors="pt").to(next(m.parameters()).device)
    n = inp["input_ids"].shape[1]
    with torch.no_grad():
        out = m.generate(
            **inp, max_new_tokens=max_new_tokens,
            temperature=0.7, top_p=0.9, do_sample=True,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0][n:], skip_special_tokens=True).strip()

PROMPT = "Explain how sparsity pruning and quantization reduce model size."

print(f"Prompt: {PROMPT}\n{'='*60}")

for label, m in [
    ("Baseline (dense fp16)", baseline_model),
    (f"SparseGPT {int(SPARSITY*100)}% + INT8", sparse_model),
]:
    print(f"\n[{label}]")
    print("-"*50)
    print(generate(m, tokenizer, PROMPT))

In [ ]:
# @title Cell 13

# @title Save Results

os.makedirs("results", exist_ok=True)
# Convert floats for JSON
def to_serializable(d):
    if isinstance(d, dict):
        return {k: to_serializable(v) for k, v in d.items()}
    if isinstance(d, (np.floating, np.float32, np.float64)):
        return float(d)
    return d

with open("results/pipeline_results.json", "w") as f:
    json.dump(to_serializable(pipeline_results), f, indent=2)

print("✅ Results saved to results/pipeline_results.json")
print("\nFinal pipeline_results:")
for stage, info in pipeline_results.items():
    print(f"\n  [{stage}]")
    for k, v in info.items():
        print(f"    {k:<25} : {v}")